In [143]:
import json
import os
import subprocess
import textwrap
from pathlib import Path
from typing import Any, cast

import httpx
from anthropic import Anthropic, omit
from anthropic.types import Message, MessageParam, TextBlock
from dotenv import dotenv_values
from IPython.display import Markdown


def initialize_env() -> None:
    env_path = Path.cwd().parent / ".env.template"
    for key, ref in dotenv_values(env_path).items():
        if ref is None:
            continue
        if not os.environ[key]:
            os.environ[key] = subprocess.run(
                ["op", "read", ref], capture_output=True, text=True, check=True
            ).stdout.strip()


def anthropic_client() -> Anthropic:
    return Anthropic(
        base_url="https://openrouter.ai/api",
        api_key=os.environ["OPENROUTER_API_KEY"],
    )


def model(name: str = "llama") -> str:
    models = {
        "deepseek": "deepseek/deepseek-v4.1-flash",
        "llama": "meta-llama/llama-3.1-8b-instruct",
        "llama-70b": "meta-llama/llama-3.3-70b-instruct",
        "llama-flagship": "meta-llama/llama-4-maverick",
        "mistral": "mistralai/mistral-small-3.1-24b-instruct",
        "haiku": "anthropic/claude-haiku-4.5",
        "qwen": "qwen/qwen-2.5-coder-32b-instruct",
    }
    return models.get(name, models["llama"])


initialize_env()
messages: list[MessageParam] = []

In [138]:
def get_reply(message: Message) -> str:
    text = next(
        (block.text for block in message.content if isinstance(block, TextBlock)),
        "",
    )
    if not text:
        return f"[no text block; stop_reason={message.stop_reason}, content={message.content!r}]"
    return text


def add_user_message(messages: list[MessageParam], text: str) -> None:
    user_message: MessageParam = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages: list[MessageParam], text: str) -> None:
    assistant_message: MessageParam = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def wrap_text(text: str, width: int = 120) -> str:
    return "\n".join(textwrap.fill(line, width=width) for line in text.splitlines())


def remaining_credits() -> float:
    response = httpx.get(
        "https://openrouter.ai/api/v1/credits",
        headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"},
        timeout=10.0,
    )
    response.raise_for_status()
    data = response.json()["data"]
    return data["total_credits"] - data["total_usage"]

In [139]:
def chat(
    messages: list[MessageParam],
    system: str | None = None,
    temperature: float = 1.0,
    stop_sequences: list[str] | None = None,
) -> str:
    params: dict[str, Any] = {
        "model": model(),
        "max_tokens": 10000,
        "thinking": {"type": "disabled"},
        "messages": messages,
        "system": system or omit,
        "temperature": temperature,
        "stop_sequences": stop_sequences or omit,
    }
    message = cast(Message, anthropic_client().messages.create(**params))
    return get_reply(message)


def chat_stream(
    messages: list[MessageParam],
    system: str | None = None,
    temperature: float = 1.0,
    stop_sequences: list[str] | None = None,
):
    params: dict[str, Any] = {
        "model": model(),
        "max_tokens": 10000,
        "thinking": {"type": "disabled"},
        "messages": messages,
        "system": system or omit,
        "temperature": temperature,
        "stop_sequences": stop_sequences or omit,
    }
    with anthropic_client().messages.stream(**params) as stream:
        for text in stream.text_stream:
            print(wrap_text(f"{text}"), end="")
    return get_reply(stream.get_final_message())

In [141]:
messages.clear()
add_user_message(messages, "Create a 1 sentence fake database description.")
Markdown(f"\n---\n{chat_stream(messages, temperature=0.999)}")

The Galactic Collection Database (GCD-Alpha) is a comprehensive repository of intergalactic records containing all known mentions of sentient life, wreckage, and cultural artifacts from over457 habitable planets and976,473 asteroid formations in the Milky Way galaxy.


---
The Galactic Collection Database (GCD-Alpha) is a comprehensive repository of intergalactic records containing all known mentions of sentient life, wreckage, and cultural artifacts from over 457 habitable planets and 976,473 asteroid formations in the Milky Way galaxy.

In [ ]:
messages.clear()

while True:
    # You: Is it true that quantum computers consume hugh amount of energy? Why is it so?
    # You: how is physical qubit made? of what, silicone?
    user_input = input("You: ")
    if user_input == "/exit":
        break

    add_user_message(messages, user_input)
    print(wrap_text(f"You: {user_input}"))

    reply = chat(
        messages,
        system="You are a scientific assistant which popularizes complex scientific concepts.",
    )
    add_assistant_message(messages, reply)

    print(wrap_text(f"AI: {reply}"))
    print("\n---\n")

In [142]:
messages.clear()
add_user_message(messages, "Generate AWS EventBridge rule as JSON")
add_assistant_message(messages, "```json\n")
print(wrap_text(chat(messages, stop_sequences=["```"]).strip()))
print(f"Remaining: ${remaining_credits():.2f}")

{
  "name": "MyEventBridgeRule",
  "eventPattern": {
    "source": ["aws.ecs-tasks"],
    "detail-type": ["ECS Task State Change"],
    "detail": {
      "state": ["RUNNING"]
    }
  },
  "state": "ENABLED",
  "detail-type-attributes": {
    "keyword": ["State"]
  },
  "targets": [
    {
      "id": "string",
      "arn": "arn:aws:events:region:account-id:rule/MyEventBridgeRule",
      "rule": "string",
      "input": "string"
    }
  ]
}
Remaining: $6.37


In [ ]:
messages.clear()
add_user_message(messages, "Generate 3 different short AWS CLI commands")
add_assistant_message(
    messages, "Here are three different short AWS CLI commands without comments:\n```bash\n"
)
print(chat(messages, stop_sequences=["```"]).strip())
Markdown(f"---\nRemaining: **${remaining_credits():.2f}**")

In [ ]:
def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages.clear()
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json\n")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)


dataset = generate_dataset()
print(dataset)
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

[{'task': 'Extract AWS instance type from a given JSON object containing instance details', 'inputs': {'instance_details': '{"InstanceType": "t2.micro", "Region": "US-East-1"}'}, 'expected_output': 't2.micro'}, {'task': 'Validate an AWS Lambda function event with a specific signature', 'inputs': {'event': '{"Records":[{"awsRegion": "us-east-1", "requestParameters": {"param1": "value1"}}]}', 'signature': 'us-east-1'}, 'expected_output': 'True'}, {'task': 'Check if a given AWS IAM user has a specific set of permissions', 'inputs': {'user': '{"User": {"Arn": "arn:aws:iam::123456789012:user/Alice", "Permissions": [\'s3:GetObject\']}}', 'permissions': ['s3:GetObject']}, 'expected_output': 'True'}]
